In [1]:
import os
import pandas as pd
import numpy as np

# Membaca dataset langsung dari file CSV
csv_filename = 'dataset_44fitur_raw.csv'

if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename, low_memory=False)
    print(f"Ukuran awal dataset: {df.shape[0]} baris dan {df.shape[1]} kolom")
    display(df.head())
else:
    print(f"File {csv_filename} tidak ditemukan. Silakan unggah file tersebut ke direktori aktif.")

Ukuran awal dataset: 579729 baris dan 45 kolom


,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,Label
0,10.055911,53.92,6.0,63.44,4.972200,2.784432,2.187768,0.16,0.24,0.0,...,358.412223,250.28,0.205223,50.0,22.896010,510.793431,8188.216450,6.684484,616.0,Benign
1,5.365657,53.92,6.0,63.40,9.318523,5.591114,3.727409,0.08,0.24,0.0,...,358.137350,265.72,0.109503,50.0,23.936026,525.914653,62161.536842,6.580101,600.0,Benign
2,3.558436,40.96,6.0,120.86,14.051116,8.149647,5.901469,0.00,0.08,0.0,...,756.141383,442.64,0.072621,50.0,29.892528,1077.963718,1437.866667,1.238542,609.0,Benign
3,3.825222,28.00,17.0,96.32,13.071137,7.319837,5.751300,0.00,0.00,0.0,...,40.320238,102.28,0.078066,50.0,14.512646,47.504504,-80.580087,52.645709,616.0,Benign
4,1.004883,28.00,17.0,87.68,49.757034,21.893095,27.863939,0.00,0.00,0.0,...,28.160852,108.92,0.020508,50.0,14.578048,23.605585,-53.805195,18.714638,616.0,Benign


In [2]:
# Menghapus spasi di awal dan akhir nama kolom
df.columns = df.columns.str.strip()

# Menampilkan daftar seluruh kolom dan tipe datanya
print("Daftar Kolom Dataset:")
print(df.dtypes)

Daftar Kolom Dataset:
flow_duration      float64
Header_Length      float64
Protocol Type      float64
Duration           float64
Rate               float64
Srate              float64
Drate              float64
fin_flag_number    float64
syn_flag_number    float64
rst_flag_number    float64
psh_flag_number    float64
ack_flag_number    float64
ece_flag_number    float64
cwr_flag_number    float64
ack_count          float64
syn_count          float64
fin_count          float64
HTTP               float64
HTTPS              float64
DNS                float64
Telnet             float64
SMTP               float64
SSH                float64
IRC                float64
TCP                float64
UDP                float64
DHCP               float64
ARP                float64
ICMP               float64
IPv                float64
LLC                float64
Tot sum            float64
Min                float64
Max                float64
AVG                float64
Std                float64
Tot si

Kolom identitas yang dihapus: []
Jumlah kolom tersisa: 45


In [4]:
# 1. Mengubah nilai Infinity dan -Infinity menjadi NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Mengecek jumlah missing values (NaN)
total_nan = df.isna().sum().sum()
print(f"Total nilai NaN/Infinity yang ditemukan: {total_nan}")

# 3. Menghapus baris yang memiliki nilai NaN
df.dropna(inplace=True)

# 4. Menghapus baris data yang duplikat
duplicate_count = df.duplicated().sum()
df.drop_duplicates(inplace=True)

print(f"Jumlah baris duplikat yang dihapus: {duplicate_count}")
print(f"Ukuran dataset setelah pembersihan NaN & duplikat: {df.shape}")

Total nilai NaN/Infinity yang ditemukan: 0
Jumlah baris duplikat yang dihapus: 64118
Ukuran dataset setelah pembersihan NaN & duplikat: (515611, 45)


In [5]:
# Mencari kolom numerik yang hanya memiliki 1 nilai unik (konstan)
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
df.drop(columns=constant_cols, inplace=True)

print(f"Kolom konstan yang dihapus: {constant_cols}")
print(f"Ukuran dataset saat ini: {df.shape}")

Kolom konstan yang dihapus: ['ARP', 'IPv', 'LLC']
Ukuran dataset saat ini: (515611, 42)


In [11]:
from sklearn.preprocessing import LabelEncoder

# Menentukan kolom target
target_col = 'Label' if 'Label' in df.columns else df.columns[-1]
print(f"Kolom target yang digunakan: {target_col}")

# Melihat distribusi kelas sebelum encoding
print("\nDistribusi Kelas Jaringan:")
print(df[target_col].value_counts())

# Encoding kolom target menjadi numerik (0, 1, 2, dst.)
le = LabelEncoder()
df[target_col] = le.fit_transform(df[target_col].astype(str))

# Menyimpan pemetaan label untuk referensi evaluasi model
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("\nPemetaan Label Target:", label_mapping)

# Memastikan seluruh kolom fitur lainnya bertipe numerik
feature_cols = [col for col in df.columns if col != target_col]
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Hapus kembali jika ada baris yang gagal dikonversi ke numerik
df.dropna(inplace=True)

Kolom target yang digunakan: Label

Distribusi Kelas Jaringan:
Label
1     111936
13     89012
4      80517
6      50978
2      38957
11     37128
3      18581
10     18090
9      17394
8      14997
0      13456
7      12596
5       7197
14      2977
12      1795
Name: count, dtype: int64

Pemetaan Label Target: {'0': np.int64(0), '1': np.int64(1), '10': np.int64(2), '11': np.int64(3), '12': np.int64(4), '13': np.int64(5), '14': np.int64(6), '2': np.int64(7), '3': np.int64(8), '4': np.int64(9), '5': np.int64(10), '6': np.int64(11), '7': np.int64(12), '8': np.int64(13), '9': np.int64(14)}


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Menyimpan seluruh dataset bersih ke file CSV baru
cleaned_filename = 'dataset_44fitur_cleaned.csv'
df.to_csv(cleaned_filename, index=False)
print(f"Dataset bersih berhasil disimpan ke: {cleaned_filename}")

# 2. Memisahkan fitur (X) dan target (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# 3. Membagi data menjadi Training (80%) dan Testing (20%) secara proporsional (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Standarisasi skala fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data siap dilatih! Ukuran X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")

Dataset bersih berhasil disimpan ke: dataset_44fitur_cleaned.csv
Data siap dilatih! Ukuran X_train: (412488, 41), X_test: (103123, 41)


In [13]:
import pandas as pd

# Memuat dataset yang telah dibersihkan
df_cleaned = pd.read_csv('dataset_44fitur_cleaned.csv')

# Menampilkan ringkasan informasi dataset bersih
print(f"Ukuran dataset bersih: {df_cleaned.shape[0]} baris dan {df_cleaned.shape[1]} kolom\n")

# Mengatur opsi tampilan pandas agar semua kolom terlihat dengan rapi
pd.set_option('display.max_columns', None)

# Menampilkan 10 baris pertama dataset
display(df_cleaned.head(10))

Ukuran dataset bersih: 515611 baris dan 42 kolom



,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,cwr_flag_number,ack_count,syn_count,fin_count,HTTP,HTTPS,DNS,Telnet,SMTP,SSH,IRC,TCP,UDP,DHCP,ICMP,Tot sum,Min,Max,AVG,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,Label
0,10.055911,53.92,6.0,63.44,4.972200,2.784432,2.187768,0.16,0.24,0.00,0.24,0.88,0.0,0.0,44.0,12.0,8.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,12514.0,66.0,1139.0,250.28,358.412223,250.28,0.205223,50.0,22.896010,510.793431,8188.216450,6.684484,616.0,0
1,5.365657,53.92,6.0,63.40,9.318523,5.591114,3.727409,0.08,0.24,0.00,0.28,0.88,0.0,0.0,44.0,12.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,13286.0,66.0,1139.0,265.72,358.137350,265.72,0.109503,50.0,23.936026,525.914653,62161.536842,6.580101,600.0,0
2,3.558436,40.96,6.0,120.86,14.051116,8.149647,5.901469,0.00,0.08,0.00,0.56,0.96,0.0,0.0,48.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,22132.0,60.0,2974.0,442.64,756.141383,442.64,0.072621,50.0,29.892528,1077.963718,1437.866667,1.238542,609.0,0
3,3.825222,28.00,17.0,96.32,13.071137,7.319837,5.751300,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,5114.0,69.0,242.0,102.28,40.320238,102.28,0.078066,50.0,14.512646,47.504504,-80.580087,52.645709,616.0,0
4,1.004883,28.00,17.0,87.68,49.757034,21.893095,27.863939,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,5446.0,77.0,175.0,108.92,28.160852,108.92,0.020508,50.0,14.578048,23.605585,-53.805195,18.714638,616.0,0
5,1.727260,43.84,6.0,101.36,28.947580,18.526451,10.421129,0.08,0.32,0.00,0.16,0.76,0.0,0.0,38.0,16.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,29300.0,60.0,2974.0,586.00,1008.852754,586.00,0.035250,50.0,39.077558,1335.275766,7007.673203,90.724967,576.0,0
6,0.057071,43.12,6.0,89.52,876.102150,420.529032,455.573118,0.16,0.26,0.08,0.20,0.92,0.0,0.0,46.0,13.0,8.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,10226.0,60.0,1678.0,204.52,358.759822,204.52,0.001165,50.0,20.128275,494.923729,-5114.659420,8.523237,624.0,0
7,0.055246,42.88,6.0,92.48,905.041020,470.621330,434.419689,0.20,0.24,0.04,0.12,0.84,0.0,0.0,42.0,12.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,32188.0,60.0,2974.0,643.76,1041.987995,643.76,0.001127,50.0,36.482170,1283.477177,72680.231884,80.948666,624.0,0
8,0.128570,41.68,6.0,86.56,388.892969,171.112906,217.780062,0.12,0.14,0.00,0.12,1.00,0.0,0.0,50.0,7.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,24224.0,60.0,2974.0,484.48,852.945584,484.48,0.002624,50.0,29.999567,1076.908526,-21288.961039,40.337568,616.0,0
9,0.076131,40.00,6.0,82.12,656.763833,249.570256,407.193576,0.00,0.00,0.00,0.08,1.00,0.0,0.0,50.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,58224.0,60.0,2974.0,1164.48,1017.318618,1164.48,0.001554,50.0,43.605268,680.691289,0.000000,0.000000,589.0,0


In [15]:
# Memuat kembali dataset hasil pembersihan
df_check = pd.read_csv('dataset_44fitur_cleaned.csv')

# 1. Mengecek nilai kosong (missing values)
null_summary = df_check.isnull().sum()
total_nulls = null_summary.sum()

# 2. Mengecek baris duplikat
total_duplicates = df_check.duplicated().sum()

print("=== HASIL PENGECEKAN ULANG DATASET BERSIH ===")
print(f"Total nilai kosong (NaN/Null) ditemukan: {total_nulls}")
if total_nulls > 0:
    print("Kolom dengan nilai kosong:")
    print(null_summary[null_summary > 0])
else:
    print("-> Hebat! Tidak ada nilai kosong di dalam dataset.")

print(f"\nTotal baris duplikat ditemukan: {total_duplicates}")
if total_duplicates > 0:
    print("-> Masih ada data duplikat yang tersisa.")
else:
    print("-> Bagus! Tidak ada data duplikat di dalam dataset.")

=== HASIL PENGECEKAN ULANG DATASET BERSIH ===
Total nilai kosong (NaN/Null) ditemukan: 0
-> Hebat! Tidak ada nilai kosong di dalam dataset.

Total baris duplikat ditemukan: 0
-> Bagus! Tidak ada data duplikat di dalam dataset.
